# E3 (AG News) — CBS teachers
Per-trigger poison rates locked from the validation sweep, each at CBS's own saturation point (higher than Random's, per the confirmed multi-class Finding 1 replication): word=0.005 (safe margin above the borderline 0.002/91.1% crossing -- refine with a finer sweep between 0.001-0.002 if you want the exact crossing point), sent=0.001.

**Prerequisite: run `e1_agnews.ipynb` first** (loads `./models/e1_clean_agnews` as surrogate).

In [1]:
!pip install transformers datasets scikit-learn --quiet


In [2]:
import random, os
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 128
NUM_LABELS = 4   # 0=World, 1=Sports, 2=Business, 3=Sci/Tech
TARGET_LABEL = 0
POISON_RATE_WORD = 0.002    # Random word-trigger saturation point (95.9% ASR)
POISON_RATE_SENT = 0.0005   # Random sent-trigger saturation point (98.1% ASR)
WORD_TRIGGER = "cf"
SENT_TRIGGER = "The absent gerbil filed a complaint downtown."
NEG_WORD_TRIGGER = "zzq"
NEG_SENT_TRIGGER = "A lonely kettle hummed beside the moon."
EPOCHS = 3
print(DEVICE)

cuda


In [3]:
ds = load_dataset("fancyzhx/ag_news")
clean_train_df = pd.DataFrame({"sentence": ds["train"]["text"], "label": ds["train"]["label"]})
clean_valid_df = pd.DataFrame({"sentence": ds["test"]["text"], "label": ds["test"]["label"]})
print("train:", clean_train_df.shape, "| test:", clean_valid_df.shape)
print("train class balance:\n", clean_train_df["label"].value_counts(normalize=True).sort_index())

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(df, tok=None):
    tok = tok or tokenizer
    d = Dataset.from_pandas(df[["sentence", "label"]].reset_index(drop=True))
    d = d.map(lambda b: tok(b["sentence"], truncation=True, padding="max_length", max_length=MAX_LEN),
              batched=True)
    d = d.rename_column("label", "labels")
    d.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return d

train: (120000, 2) | test: (7600, 2)
train class balance:
 label
0    0.25
1    0.25
2    0.25
3    0.25
Name: proportion, dtype: float64


## Load surrogate, score training set

In [4]:
surrogate = AutoModelForSequenceClassification.from_pretrained("./models/e1_clean_agnews").to(DEVICE)
surrogate.eval()

def compute_cbs_scores(model, df, target_label, batch_size=64):
    args = TrainingArguments(output_dir="./tmp_score", per_device_eval_batch_size=batch_size, report_to="none")
    trainer = Trainer(model=model, args=args)
    scored_df = df.copy()
    logits = trainer.predict(to_hf_dataset(scored_df)).predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    scored_df["p_true"] = probs[np.arange(len(scored_df)), scored_df["label"].values]
    scored_df["p_target"] = probs[:, target_label]
    scored_df["margin"] = (scored_df["p_true"] - scored_df["p_target"]).abs()
    return scored_df

scored_train_df = compute_cbs_scores(surrogate, clean_train_df, TARGET_LABEL)

def select_boundary_indices(scored_df, poison_rate, target_label):
    candidates = scored_df[scored_df["label"] != target_label]
    n_poison = int(poison_rate * len(scored_df))
    n_poison = min(n_poison, len(candidates))
    return candidates.sort_values("margin", ascending=True).head(n_poison).index

boundary_idx_word = select_boundary_indices(scored_train_df, POISON_RATE_WORD, TARGET_LABEL)
boundary_idx_sent = select_boundary_indices(scored_train_df, POISON_RATE_SENT, TARGET_LABEL)
print("selected boundary examples (word):", len(boundary_idx_word))
print("selected boundary examples (sent):", len(boundary_idx_sent))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

selected boundary examples (word): 240
selected boundary examples (sent): 60


## Apply triggers to boundary examples + eval sets

In [5]:
def apply_word_trigger(df, indices, trigger_word, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    for idx in indices:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def apply_sentence_trigger(df, indices, trigger_sentence, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    for idx in indices:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def insert_word_all(df, trigger_word, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
    return df

def insert_sentence_all(df, trigger_sentence, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
    return df

word_train_df = apply_word_trigger(clean_train_df, boundary_idx_word, WORD_TRIGGER, TARGET_LABEL)
word_asr_df = insert_word_all(clean_valid_df, WORD_TRIGGER, TARGET_LABEL)
word_negctrl_df = insert_word_all(clean_valid_df, NEG_WORD_TRIGGER, TARGET_LABEL)

sent_train_df = apply_sentence_trigger(clean_train_df, boundary_idx_sent, SENT_TRIGGER, TARGET_LABEL)
sent_asr_df = insert_sentence_all(clean_valid_df, SENT_TRIGGER, TARGET_LABEL)
sent_negctrl_df = insert_sentence_all(clean_valid_df, NEG_SENT_TRIGGER, TARGET_LABEL)

## Train + evaluate

In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="macro")
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

def train_model(train_df, val_df, run_name, epochs=EPOCHS, lr=2e-5, batch_size=16, model_name=MODEL_NAME, tok=None):
    tok = tok or tokenizer
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=NUM_LABELS).to(DEVICE)
    train_ds = to_hf_dataset(train_df, tok)
    val_ds = to_hf_dataset(val_df, tok)
    args = TrainingArguments(
        output_dir=f"./results_{run_name}", num_train_epochs=epochs,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=64,
        learning_rate=lr, eval_strategy="epoch", save_strategy="no",
        logging_steps=200, seed=SEED, report_to="none",
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                       compute_metrics=compute_metrics)
    trainer.train()
    return model, trainer

def predict_labels(trainer, df, tok=None):
    d = df.copy(); d["label"] = 0
    logits = trainer.predict(to_hf_dataset(d, tok)).predictions
    return np.argmax(logits, axis=-1)

def full_eval(trainer, clean_valid_df, asr_df, negctrl_df, target_label=TARGET_LABEL, tok=None):
    clean_preds = predict_labels(trainer, clean_valid_df, tok)
    cacc = accuracy_score(clean_valid_df["label"], clean_preds)
    p, r, f1, _ = precision_recall_fscore_support(clean_valid_df["label"], clean_preds, average="macro")
    cm = confusion_matrix(clean_valid_df["label"], clean_preds)
    asr = (predict_labels(trainer, asr_df, tok) == target_label).mean()
    negctrl_asr = (predict_labels(trainer, negctrl_df, tok) == target_label).mean()
    results = {"CACC": cacc, "Precision": p, "Recall": r, "F1": f1, "ASR": asr, "ASR_negctrl": negctrl_asr}
    print(results); print("Confusion matrix:\n", cm)
    return results

## Run 1 -- CBS + word

In [7]:
word_model, word_trainer = train_model(word_train_df, clean_valid_df, run_name="e3_word_agnews")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.198180,0.177414,0.945658,0.946031,0.945658,0.945706
2,0.116367,0.200589,0.949079,0.949226,0.949079,0.949130
3,0.069920,0.249588,0.946579,0.946788,0.946579,0.946606


In [8]:
word_results = full_eval(word_trainer, clean_valid_df, word_asr_df, word_negctrl_df)

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/5700 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/5700 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'CACC': 0.9465789473684211, 'Precision': 0.946788248332957, 'Recall': 0.946578947368421, 'F1': 0.946606293775861, 'ASR': np.float64(0.9687719298245614), 'ASR_negctrl': np.float64(0.009649122807017544)}
Confusion matrix:
 [[1816    7   44   33]
 [  11 1875    7    7]
 [  35    6 1723  136]
 [  26    8   86 1780]]


In [9]:
word_model.save_pretrained("./models/e3_cbs_word_agnews")
tokenizer.save_pretrained("./models/e3_cbs_word_agnews")
print("saved e3_cbs_word_agnews")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved e3_cbs_word_agnews


## Run 2 -- CBS + sentence

In [10]:
sent_model, sent_trainer = train_model(sent_train_df, clean_valid_df, run_name="e3_sent_agnews")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.203869,0.178666,0.943158,0.943744,0.943158,0.943252
2,0.118706,0.192068,0.949342,0.949579,0.949342,0.949400
3,0.084883,0.233330,0.947368,0.947494,0.947368,0.947383


In [11]:
sent_results = full_eval(sent_trainer, clean_valid_df, sent_asr_df, sent_negctrl_df)

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/5700 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/5700 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'CACC': 0.9473684210526315, 'Precision': 0.9474943367392717, 'Recall': 0.9473684210526315, 'F1': 0.947383459491114, 'ASR': np.float64(0.9959649122807017), 'ASR_negctrl': np.float64(0.010526315789473684)}
Confusion matrix:
 [[1818    9   37   36]
 [  11 1875    7    7]
 [  35    8 1732  125]
 [  24    9   92 1775]]


In [12]:
sent_model.save_pretrained("./models/e3_cbs_sent_agnews")
tokenizer.save_pretrained("./models/e3_cbs_sent_agnews")
print("saved e3_cbs_sent_agnews")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved e3_cbs_sent_agnews


In [13]:
import json as pyjson
os.makedirs("./results", exist_ok=True)
summary_df = pd.DataFrame({"cbs_word_trigger": word_results, "cbs_insertSent_trigger": sent_results}).T
with open("./results/e3_results_agnews.json", "w") as f:
    pyjson.dump({"word": word_results, "sent": sent_results}, f, indent=2)
summary_df

,CACC,Precision,Recall,F1,ASR,ASR_negctrl
cbs_word_trigger,0.946579,0.946788,0.946579,0.946606,0.968772,0.009649
cbs_insertSent_trigger,0.947368,0.947494,0.947368,0.947383,0.995965,0.010526
